# e02d_vit_5 — train

이 Notebook은 이 실험만 학습합니다. GPU·인터넷 ON, 대회 데이터를 연결하고 Run All 하세요. OOF 평가 후 전체 gold로 최종 모델을 만들고 run/bundle에 저장합니다.


In [ ]:
EXPERIMENT = {'id': 'e02d_vit_5', 'backbone': 'vit_b16', 'feature_dim': 768, 'slices': 5, 'crop': 224, 'resize': 256, 'interpolation': 'bilinear', 'aggregation': 'mean', 'position': False, 'plane_embedding': False, 'epochs': 5, 'lr': 0.001, 'weight_decay': 0.1, 'pipeline_sha256': 'd022cba4ae8327be341305089b4051173a461bd35d5c15a84de793c66d5e9cb5'}


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pydicom==3.0.2', 'python-gdcm==3.2.6'])


In [ ]:
"""Shared source embedded into each standalone experiment notebook."""

from pathlib import Path

import hashlib

import json

import math

import numpy as np

import pandas as pd

import torch

from torch import nn

PLANES = ("Sagittal", "Coronal", "Axial")

LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]

SEEDS = (42, 2026, 3407)

def file_hash(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def validate_bundle(path, expected_config):
    path = Path(path).resolve()
    manifest = json.loads((path / "assets.json").read_text(encoding="utf-8"))
    if manifest["experiment"] != expected_config:
        raise ValueError("Wrong experiment bundle: attach this experiment's completed training version.")
    files = manifest["files"]
    wheels = {p.relative_to(path).as_posix() for p in (path / "wheels").glob("*.whl")}
    declared_wheels = {name for name in files if name.startswith("wheels/") and name.endswith(".whl")}
    if (not {"encoder.ts", "model.pt"}.issubset(files) or wheels != declared_wheels
            or len(wheels) != 2
            or not any(name.startswith("wheels/pydicom-3.0.2-") for name in wheels)
            or not any(name.startswith("wheels/python_gdcm-3.2.6-") for name in wheels)):
        raise ValueError("Bundle must include hashes for both models and exactly the two required wheels.")
    for name, digest in manifest["files"].items():
        target = (path / name).resolve()
        if not target.is_relative_to(path) or not target.is_file() or file_hash(target) != digest:
            raise ValueError(f"Invalid or corrupted bundle file: {name}")
    return manifest

def find_data(input_root=Path("/kaggle/input")):
    for path in (input_root / "rsna-knee-abnormality-detection",
                 input_root / "competitions/rsna-knee-abnormality-detection"):
        if (path / "test.csv").is_file():
            return path
    raise FileNotFoundError("Attach the RSNA Knee competition data.")

def find_bundle(input_root=Path("/kaggle/input")):
    paths = list(input_root.glob("*/run/bundle/assets.json"))
    paths += list(input_root.glob("notebooks/*/*/run/bundle/assets.json"))
    paths += list(input_root.glob("kernels/*/*/run/bundle/assets.json"))
    if len(paths) != 1:
        raise ValueError("Attach exactly one completed experiment training output.")
    return paths[0].parent

def make_transform(config):
    from torchvision.transforms import InterpolationMode
    from torchvision.transforms._presets import ImageClassification
    return ImageClassification(crop_size=config["crop"], resize_size=config["resize"],
                               interpolation=InterpolationMode(config["interpolation"]), antialias=True)

def ordered_slices(folder):
    import pydicom
    paths = sorted(folder.glob("*.dcm"))
    if not paths:
        raise FileNotFoundError(f"No DICOM slices in {folder}")
    headers = [pydicom.dcmread(p, stop_before_pixels=True,
               specific_tags=["ImagePositionPatient", "ImageOrientationPatient", "InstanceNumber"]) for p in paths]
    if all(hasattr(h, "ImagePositionPatient") and hasattr(h, "ImageOrientationPatient") for h in headers):
        orientation = np.asarray(headers[0].ImageOrientationPatient, dtype=float)
        normal = np.cross(orientation[:3], orientation[3:])
        if np.linalg.norm(normal) <= .9:
            raise ValueError("Invalid DICOM orientation.")
        coordinates = np.asarray([np.dot(np.asarray(h.ImagePositionPatient, dtype=float), normal) for h in headers])
    elif all(hasattr(h, "InstanceNumber") for h in headers):
        coordinates = np.asarray([float(h.InstanceNumber) for h in headers])
    else:
        raise ValueError(f"No slice position or InstanceNumber: {folder}")
    if not np.isfinite(coordinates).all():
        raise ValueError("Non-finite slice positions.")
    order = np.argsort(coordinates, kind="stable")
    coordinates = coordinates[order]
    span = np.ptp(coordinates)
    positions = (coordinates - coordinates.min()) / span if span > 0 else np.full(len(paths), .5)
    return [paths[i] for i in order], positions.astype(np.float32)

def image_tensor(path, transform):
    import pydicom
    ds = pydicom.dcmread(path)
    pixels = ds.pixel_array.astype(np.float32)
    if pixels.ndim != 2 or not np.isfinite(pixels).all():
        raise ValueError(f"Invalid MRI slice: {path}")
    pixels = pixels * float(getattr(ds, "RescaleSlope", 1)) + float(getattr(ds, "RescaleIntercept", 0))
    if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
        pixels = -pixels
    low, high = np.percentile(pixels, [1, 99])
    if high <= low:
        low, high = float(pixels.min()), float(pixels.max())
    if high <= low:
        raise ValueError(f"Constant MRI slice: {path}")
    pixels = np.clip((pixels - low) / (high - low), 0, 1)
    return transform(torch.from_numpy(pixels).float().unsqueeze(0).repeat(3, 1, 1))

@torch.inference_mode()
def study_tokens(uid, series, image_root, encoder, transform, config, device):
    count, dim = config["slices"], config["feature_dim"]
    features = torch.zeros(3, count, dim)
    mask = torch.zeros(3, count, dtype=torch.bool)
    positions = torch.zeros(3, count)
    rows = series.loc[series.StudyInstanceUID == uid]
    for plane_index, plane in enumerate(PLANES):
        choices = rows.loc[rows.Anatomical_Plane == plane].sort_values(
            ["Fluid_Sensitive", "Fat_Suppression", "SeriesInstanceUID"], ascending=[False, False, True])
        if choices.empty:
            continue
        paths, coords = ordered_slices(image_root / uid / choices.iloc[0].SeriesInstanceUID)
        indices = np.unique(np.rint(np.linspace(.25, .75, count) * (len(paths) - 1)).astype(int))
        batch = torch.stack([image_tensor(paths[i], transform) for i in indices]).to(device)
        output = encoder(batch).cpu().float()
        if output.shape != (len(indices), dim) or not torch.isfinite(output).all():
            raise ValueError("Unexpected encoder features.")
        features[plane_index, :len(indices)] = output
        mask[plane_index, :len(indices)] = True
        positions[plane_index, :len(indices)] = torch.from_numpy(coords[indices])
    if not mask.any():
        raise ValueError(f"No supported MRI planes for study {uid}")
    return features, mask, positions

def pooled_features(tokens, mask):
    clean = torch.where(mask[..., None], tokens, 0.)
    return (clean.sum(2) / mask.sum(2).clamp_min(1)[..., None]).flatten(1)

class StudyHead(nn.Module):
    def __init__(self, config, mean, std):
        super().__init__()
        self.mode = config["aggregation"]
        self.position = config["position"]
        self.register_buffer("mean", mean.clone())
        self.register_buffer("std", std.clone())
        if self.mode == "mean":
            self.classifier = nn.Linear(3 * config["feature_dim"], 12)
        elif self.mode == "attention":
            self.project = nn.Linear(config["feature_dim"], 64)
            layer = nn.TransformerEncoderLayer(64, 4, 128, dropout=0., batch_first=True, norm_first=True)
            self.attention = nn.TransformerEncoder(layer, 1, enable_nested_tensor=False)
            self.classifier = nn.Linear(3 * 64, 12)
            # Initialize common layers identically before adding optional plane parameters.
            self.plane_embedding = nn.Parameter(torch.zeros(3, 64)) if config["plane_embedding"] else None
            self.register_buffer("frequencies", torch.exp(torch.arange(0, 64, 2) * (-math.log(10000.) / 64)))
        else:
            raise ValueError(f"Unknown aggregation: {self.mode}")

    def forward(self, tokens, mask, positions):
        if self.mode == "mean":
            return self.classifier((pooled_features(tokens, mask) - self.mean) / self.std)
        clean = torch.where(mask[..., None], tokens, 0.)
        x = self.project((clean - self.mean) / self.std)
        if self.position:
            angles = positions[..., None] * (2 * math.pi) * self.frequencies
            x = x + torch.stack((angles.sin(), angles.cos()), dim=-1).flatten(-2)
        if self.plane_embedding is not None:
            x = x + self.plane_embedding[None, :, None, :]
        b, planes, slices, dim = x.shape
        flat_mask = mask.reshape(b * planes, slices)
        safe_mask = flat_mask.clone()
        safe_mask[~safe_mask.any(1), 0] = True
        x = self.attention(x.reshape(b * planes, slices, dim), src_key_padding_mask=~safe_mask)
        x = torch.where(flat_mask[..., None], x, 0.)
        vectors = x.sum(1) / flat_mask.sum(1).clamp_min(1)[:, None]
        return self.classifier(vectors.reshape(b, planes * dim))

def fit_model(tokens, mask, positions, y, train_idx, seed, config):
    torch.manual_seed(seed)
    x, m, p, target = tokens[train_idx], mask[train_idx], positions[train_idx], y[train_idx]
    statistics = pooled_features(x, m) if config["aggregation"] == "mean" else x[m]
    mean, std = statistics.mean(0), statistics.std(0).clamp_min(.01)
    positives = target.sum(0)
    if not torch.isfinite(statistics).all() or not ((positives > 0) & (positives < len(target))).all():
        raise ValueError("Finite training features and both classes per label are required.")
    head = StudyHead(config, mean, std)
    optimizer = torch.optim.AdamW(head.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
    criterion = nn.BCEWithLogitsLoss(pos_weight=(len(target) - positives) / positives)
    history = []
    head.train()
    for epoch in range(1, config["epochs"] + 1):
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(head(x, m, p), target)
        if not torch.isfinite(loss):
            raise ValueError("Non-finite training loss.")
        loss.backward()
        optimizer.step()
        history.append({"epoch": epoch, "train_loss": loss.item()})
    return head.eval(), history

def make_folds(ids, y):
    from sklearn.model_selection import KFold
    if len(set(ids)) != len(ids) or len(y) != len(ids) or not np.isin(y, [0, 1]).all():
        raise ValueError("Expected unique IDs and complete binary labels.")
    order = np.argsort(ids)
    folds = np.full(len(ids), -1, dtype=int)
    for fold, (train, valid) in enumerate(KFold(3, shuffle=True, random_state=42).split(order)):
        for index in (order[train], order[valid]):
            count = y[index].sum(0)
            if not ((count > 0) & (count < len(index))).all():
                raise ValueError("Each fold/label needs positives and negatives.")
        folds[order[valid]] = fold
    return folds

def metric_scores(y, logits):
    from sklearn.metrics import roc_auc_score
    probabilities = logits.sigmoid().numpy()
    aucs = roc_auc_score(y, probabilities, average=None)
    return {"macro_auc": float(np.mean(aucs)), "per_label_auc": dict(zip(LABELS, aucs.tolist())),
            "bce": nn.functional.binary_cross_entropy_with_logits(logits, torch.from_numpy(y)).item()}

def save_submission(ids, probabilities, sample_columns, target):
    probabilities = np.asarray(probabilities)
    if (not ids or len(set(ids)) != len(ids) or list(sample_columns) != ["StudyInstanceUID", *LABELS]
            or probabilities.shape != (len(ids), 12) or not np.isfinite(probabilities).all()
            or not ((probabilities >= 0) & (probabilities <= 1)).all()):
        raise ValueError("Invalid submission IDs, label order, shape or probabilities.")
    frame = pd.DataFrame(probabilities, columns=LABELS)
    frame.insert(0, "StudyInstanceUID", ids)
    target = Path(target)
    temporary = target.with_suffix(".csv.part")
    frame.to_csv(temporary, index=False)
    temporary.replace(target)
    return frame


In [ ]:
def build_encoder(pretrained=True):
    from torchvision.models import ViT_B_16_Weights, vit_b_16
    model = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1 if pretrained else None)
    model.heads = nn.Identity()
    return model


In [ ]:
"""Training entry point; embedded after the core and one backbone constructor."""

import importlib.metadata

import platform

import subprocess

import sys

import time

def export_encoder(encoder, config, target):
    torch.backends.mha.set_fastpath_enabled(False)
    encoder = encoder.cpu().eval().requires_grad_(False)
    size = config["crop"]
    example = torch.zeros(1, 3, size, size)
    with torch.no_grad():
        # Fixed H/W is part of the bundle contract; batch sizes are independently checked below.
        traced = torch.jit.trace(encoder, example, check_trace=False)
        for batch_size in range(1, config["slices"] + 1):
            probe = torch.rand(batch_size, 3, size, size)
            actual, expected = traced(probe), encoder(probe)
            print(f"Encoder export batch={batch_size}: max_abs_error={(actual - expected).abs().max().item():.8g}", flush=True)
            # Float32 eager/JIT execution can round differently near zero, as observed in DINOv2.
            torch.testing.assert_close(actual, expected, rtol=1e-4, atol=1e-4)
    traced.save(str(target))
    loaded = torch.jit.load(str(target), map_location="cpu").eval()
    with torch.no_grad():
        torch.testing.assert_close(loaded(probe), encoder(probe), rtol=1e-4, atol=1e-4)
    return loaded

def evaluate_cv(tokens, mask, positions, y, ids, folds, config, out):
    runs, histories, predictions = [], [], []
    for seed in SEEDS:
        oof = torch.full((len(ids), 12), float("nan"))
        fold_scores = []
        for fold in range(3):
            train_index, valid_index = np.flatnonzero(folds != fold), np.flatnonzero(folds == fold)
            head, history = fit_model(tokens, mask, positions, torch.from_numpy(y), train_index, seed, config)
            with torch.inference_mode():
                logits = head(tokens[valid_index], mask[valid_index], positions[valid_index])
            oof[valid_index] = logits
            fold_scores.append({"fold": fold, **metric_scores(y[valid_index], logits)})
            histories.extend({"seed": seed, "fold": fold, **row} for row in history)
        if not torch.isfinite(oof).all():
            raise ValueError("Incomplete OOF predictions.")
        runs.append({"seed": seed, **metric_scores(y, oof), "folds": fold_scores})
        prediction = pd.DataFrame(oof.sigmoid().numpy(), columns=LABELS)
        prediction.insert(0, "fold", folds)
        prediction.insert(0, "seed", seed)
        prediction.insert(0, "StudyInstanceUID", ids)
        predictions.append(prediction)
    metrics = {"experiment": config, "runs": runs,
               "macro_auc_mean": float(np.mean([r["macro_auc"] for r in runs])),
               "macro_auc_seed_std": float(np.std([r["macro_auc"] for r in runs])),
               "fold_auc_mean": float(np.mean([f["macro_auc"] for r in runs for f in r["folds"]])),
               "bce_mean": float(np.mean([r["bce"] for r in runs])),
               "note": "Development CV, 58 studies; seed SD is not a confidence interval. Final refit is not validation."}
    (out / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    pd.concat(predictions, ignore_index=True).to_csv(out / "oof_predictions.csv", index=False)
    pd.DataFrame(histories).to_csv(out / "history.csv", index=False)
    return metrics

def train_experiment(config, encoder_factory, data, out):
    from tqdm.auto import tqdm
    out, data = Path(out), Path(data)
    if out.exists() and any(out.iterdir()):
        raise FileExistsError("Choose an empty training output directory; previous results are preserved.")
    out.mkdir(parents=True, exist_ok=True)
    bundle = out / "bundle"
    bundle.mkdir()
    torch.set_num_threads(1)
    torch.manual_seed(42)
    if not torch.cuda.is_available():
        raise RuntimeError("Run training notebook with a Kaggle GPU.")
    device = torch.device("cuda")
    train = pd.read_csv(data / "train.csv")
    series = pd.read_csv(data / "train_series.csv")
    sample = pd.read_csv(data / "sample_submission.csv", nrows=0)
    assert sample.columns.tolist() == ["StudyInstanceUID", *LABELS]
    assert train.StudyInstanceUID.is_unique and series.SeriesInstanceUID.is_unique
    assert set(series.StudyInstanceUID) <= set(train.StudyInstanceUID)
    assert train[LABELS].stack().dropna().isin([0, 1]).all()
    gold = train.loc[train[LABELS].notna().all(axis=1)].sort_values("StudyInstanceUID").reset_index(drop=True)
    assert len(gold) == 58, "Gold population changed; review the fixed experiment protocol."
    ids, y = gold.StudyInstanceUID.tolist(), gold[LABELS].to_numpy(dtype=np.float32)
    folds = make_folds(ids, y)
    pd.DataFrame({"StudyInstanceUID": ids, "fold": folds}).to_csv(out / "folds.csv", index=False)
    encoder = export_encoder(encoder_factory(), config, bundle / "encoder.ts").to(device)
    transform = make_transform(config)
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    rows = [study_tokens(uid, series, data / "train_series", encoder, transform, config, device)
            for uid in tqdm(ids, desc=config["id"] + " MRI tokens")]
    tokens, mask, positions = [torch.stack([row[i] for row in rows]).clone() for i in range(3)]
    torch.cuda.synchronize()
    timing = {"feature_seconds": time.perf_counter() - started,
              "peak_cuda_allocated_mb": torch.cuda.max_memory_allocated() / 1024**2,
              "gpu": torch.cuda.get_device_name(0), "images": int(mask.sum()),
              "note": "Single-run extraction timing; not a warmed repeated latency benchmark."}
    torch.save({"tokens": tokens, "mask": mask, "positions": positions, "study_ids": ids}, out / "tokens.pt")
    metrics = evaluate_cv(tokens, mask, positions, y, ids, folds, config, out)
    final_head, history = fit_model(tokens, mask, positions, torch.from_numpy(y), np.arange(len(ids)), 42, config)
    torch.save({"state": final_head.state_dict(), "experiment": config, "labels": LABELS,
                "train_ids": ids, "seed": 42}, bundle / "model.pt")
    pd.DataFrame(history).to_csv(out / "final_fit_history.csv", index=False)
    wheels = bundle / "wheels"
    wheels.mkdir()
    subprocess.check_call([sys.executable, "-m", "pip", "download", "-q", "--only-binary=:all:",
                           "--no-deps", "--dest", str(wheels), "pydicom==3.0.2", "python-gdcm==3.2.6"])
    manifest = {"experiment": config, "labels": LABELS, "planes": list(PLANES),
                "python": list(sys.version_info[:2]), "torch": torch.__version__,
                "packages": {p: importlib.metadata.version(p) for p in
                             ["torchvision", "numpy", "pandas", "scikit-learn", "pydicom", "python-gdcm"]},
                "final_training_studies": len(ids), "final_seed": 42,
                "folds_sha256": file_hash(out / "folds.csv"), "train_csv_sha256": file_hash(data / "train.csv"),
                "head_parameters": sum(p.numel() for p in final_head.parameters()),
                "files": {p.relative_to(bundle).as_posix(): file_hash(p) for p in sorted(bundle.rglob("*")) if p.is_file()}}
    (bundle / "assets.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    (out / "timings.json").write_text(json.dumps(timing, indent=2), encoding="utf-8")
    print(json.dumps({"id": config["id"], "macro_auc_mean": metrics["macro_auc_mean"],
                      "fold_auc_mean": metrics["fold_auc_mean"], "bce_mean": metrics["bce_mean"],
                      "timing": timing, "final_train_studies": len(ids)}, indent=2), flush=True)
    return metrics


In [ ]:
train_experiment(EXPERIMENT, build_encoder, find_data(), Path('/kaggle/working/run'))
